# Decoding the Rhythms of Emotion: a Sentimental Journey Through Music Genres
## Notebook 2 of 4: Initial Preprocessing

**Text Mining 2026/2027 | Group XX**
Raya Daher Alkhatib (20241416), Danyial Ahmad (20241831), Shaahir Racharla (20241419), Pedro Ambar (20231709)

| Notebook | Content |
|---|---|
| 01_eda | Abstract, Introduction, Data Exploration |
| **02_preprocessing** | Initial Preprocessing (writes the cleaned data) |
| 03_genre_classification | Genre Identification (text classification + Kaggle submission) |
| 04_sentiment_analysis | Sentiment Analysis, Conclusion |

Run order: 01 → 02 → 03/04. Notebook 1 only explores the data; notebook 2 writes `train_clean.pkl` and `test_clean.pkl`,
which notebooks 3 and 4 read. All raw files (`train.csv`, `test.csv`, `sample_submission.csv`) are expected in a `data/` folder next to the notebooks.

The choices below are motivated by the findings of notebook 1 (01_eda).

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path


_candidates = [Path("data"), Path("../data"), Path(".")]
DATA = next((p for p in _candidates if (p / "train.csv").exists()), None)
if DATA is None:
    raise FileNotFoundError("train.csv not found. Put train.csv and test.csv in a folder called 'data' "
                            "in the same folder as this notebook. Current folder: " + str(Path.cwd()))
print("Reading data from:", DATA)

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
print("train:", train.shape, "| test:", test.shape)

Reading data from: data
train: (134967, 8) | test: (33742, 7)


## Initial Preprocessing

**Decisions and rationale**

| Step | Decision | Why |
|---|---|---|
| Lowercasing | Yes | "Love" and "love" are the same word for our purpose. |
| Section tags `[Chorus: X]` | Converted to a single token (`sec_chorus`, `sec_hook`, ...); the performer name inside the tag is dropped | EDA showed the tag type differs by genre; artist names would just be noise / leakage. |
| Other bracketed text (stage directions, `[?]`), HTML entities (`&nbsp;`) | Removed | Not lyrics. |
| Punctuation, digits | Removed, apostrophes kept | Keeps contractions like *don't*, *ain't*, *gon'*, which are stylistic markers of genre. |
| Stopwords | **Kept** for classification, removed only for word-frequency displays | Pronouns and function words (*i, you, we, ya, ain't*) are style signals; TF-IDF already down-weights very frequent words. We compare both options in notebook 3. For VADER sentiment, raw text is needed (it uses negations like *not*, *never*). |
| Stemming / lemmatization | Not applied | Lyrics are full of slang and phonetic spellings (*lovin'*, *gotta*, *wanna*) that lemmatizers handle badly; with 135k songs the vocabulary is large enough that TF-IDF n-grams work without it. |
| Song sections | Kept the full song as one document for classification; additionally split **chorus** vs **verse** text for the sentiment analysis | Classification benefits from all the text; for sentiment it's interesting to compare the chorus (the "message") with the verses. |
| Text → numbers | Done in notebook 3 (TF-IDF) | So it is fit on training folds only. |

In [ ]:
SECTION_MAP = {
    "chorus": "chorus", "hook": "hook", "verse": "verse", "pre-chorus": "prechorus", "pre chorus": "prechorus",
    "post-chorus": "postchorus", "bridge": "bridge", "outro": "outro", "intro": "intro", "refrain": "refrain",
    "interlude": "interlude", "drop": "drop", "instrumental": "instrumental", "guitar solo": "solo",
    "solo": "solo", "breakdown": "breakdown", "break": "breakdown", "pre-hook": "prehook",
}

def section_token(match):
    inside = match.group(1).lower().split(":")[0]
    inside = re.sub(r"[\d\(\)x]+$", "", inside).strip()
    inside = re.sub(r"\s*\d+$", "", inside).strip()
    for key, tok in SECTION_MAP.items():
        if inside.startswith(key):
            return f" sec_{tok} "
    return " "

def clean_lyrics(text):
    text = re.sub(r"\[([^\]]*)\]", section_token, text)  
    text = re.sub(r"&\w+;|\bnbsp\b", " ", text)             
    text = text.lower()
    text = text.replace("’", "'").replace("‘", "'")
    text = re.sub(r"[^a-z_'\s]", " ", text)                  
    text = re.sub(r"(?<![a-z])'|'(?![a-z])", " ", text)    
    return re.sub(r"\s+", " ", text).strip()

print(clean_lyrics(train.loc[1, "lyrics"])[:400])

kid naruto rap sec_hook everybody wants you to hurt everybody wants you to cry with your face in the dirt gotta put in time gotta work everybody's praying that i fall but i gotta make it work everybody wants you to hurt everybody wants you to cry with your face in the dirt gotta put in time gotta work everybody's praying that i fall but i gotta make it sec_verse hoping it might work out one day wr


For the sentiment analysis we also want the **raw** text (not lowercased, punctuation and line breaks kept), because VADER uses
capitalisation and punctuation (e.g. "GREAT!!!") as intensity cues and we will score each lyric line separately. We also keep
choruses and verses separately.

In [3]:
def split_sections(text):
    """Return (chorus_text, verse_text) using the [..] tags. Songs without tags return ('', '')."""
    parts = re.split(r"\[([^\]]*)\]", text)
    chorus, verse = [], []
    for tag, body in zip(parts[1::2], parts[2::2]):
        t = tag.lower()
        if "chorus" in t or "hook" in t or "refrain" in t:
            chorus.append(body)
        elif "verse" in t:
            verse.append(body)
    return "\n".join(chorus).strip(), "\n".join(verse).strip()

def raw_for_sentiment(text):
    text = re.sub(r"\[[^\]]*\]|&\w+;", " ", text)        
    text = re.sub(r"[ \t]+", " ", text)                    
    return re.sub(r"\n\s*\n+", "\n", text).strip()

ch, ve = split_sections(train.loc[1, "lyrics"])
print("CHORUS:", ch[:150], "\nVERSE:", ve[:150])

CHORUS: Everybody wants you to hurt
Everybody wants you to cry with your face In the dirt
Gotta put in time, gotta work
Everybody’s praying that I fall
But I  
VERSE: Hoping it might work out one day
Wrong
Do it yourself or you might as well fall
Gotta do the grind
If you never wanna fall
Better Aim for the top or y


In [4]:
def preprocess(df):
    out = df.copy()
    out["clean"] = out["lyrics"].map(clean_lyrics)
    out["raw_text"] = out["lyrics"].map(raw_for_sentiment)
    sections = out["lyrics"].map(split_sections)
    out["chorus_text"] = sections.str[0]
    out["verse_text"] = sections.str[1]
    out["year_clean"] = out["year"].where(out["year"] >= 1900)       
    out["log_views"] = np.log1p(out["views"])
    out["n_words"] = out["lyrics"].str.split().str.len()
    return out

train_clean = preprocess(train.drop_duplicates(subset="lyrics", keep="first"))
test_clean = preprocess(test)
print("train after removing duplicate lyrics:", train_clean.shape, "| test:", test_clean.shape)
print("Empty documents after cleaning:", (train_clean["clean"].str.len() == 0).sum(), (test_clean["clean"].str.len() == 0).sum())
train_clean[["tag", "clean"]].head(3)

train after removing duplicate lyrics: (134789, 15) | test: (33742, 14)
Empty documents after cleaning: 1 1


,tag,clean
0,rock,when you said you loved me did you mean it the...
1,rap,kid naruto rap sec_hook everybody wants you to...
2,pop,sec_verse people tell me i've changed i find i...


Note that we never drop rows from the **test** set: every test id needs a prediction.

In [6]:
cols = ["id", "title", "artist", "year", "year_clean", "views", "log_views", "features", "n_words",
        "clean", "raw_text", "chorus_text", "verse_text"]
train_clean[cols + ["tag"]].to_pickle(DATA / "train_clean.pkl")
test_clean[cols].to_pickle(DATA / "test_clean.pkl")
print("Saved data/train_clean.pkl and data/test_clean.pkl")

Saved data/train_clean.pkl and data/test_clean.pkl
